In [ ]:
# FOLLOW-UP DURATION ANALYSIS — ACTIVE vs NON-DIALA; because the results were odd to both my PI and me, I wanted to check if the follow-up duration was different between the two groups, and if it was influencing the outcome (delta HbA1c). This is important because if the follow-up duration is significantly different between groups, it could confound the results of the main analysis.
# Active sheet:    Followup_Duration_BL_FU already in MONTHS
# Non-DiaLA sheet: Baseline_Date and FU_DATE → subtract → convert to months

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

replace_vals = ["", " ", "NA", "N/A", "na", "n/a", "NIL", "nil",
                "None", "none", "NULL", "null", "-", "--", "nan"]

kupp_map = {
    "Politician": 10, "Manager": 10, "Central Government Service": 10,
    "State Government Service": 10, "Tahsildar": 10,
    "Doctor": 9, "Doctor-Dentist": 9, "Doctor-General Physician": 9,
    "Doctor-Ophthalmologist": 9, "Doctor-Homeopathist": 9,
    "Doctor-Pediatrician": 9, "Doctor-Gynecologist": 9,
    "Doctor-Surgeon": 9, "Doctor-Neurologist": 9,
    "Doctor-Siddha": 9, "Doctor-Ayurvedic": 9,
    "Advocate": 9, "Architecture": 9, "AUDITOR": 9,
    "Engineer": 9, "IT Professional": 9,
    "Professor / Teacher / Education": 9, "Doctorate": 9,
    "Reporter": 8, "Accounts/Finance": 8, "Bank": 8,
    "IT Employee": 8, "Supervisor": 8, "Armed Forces": 8, "Police": 8,
    "Clerk": 7, "Government": 7,
    "Business": 6, "Self Employed": 6, "Fashion / Saloon": 6,
    "Retired Employee": 6, "Father In Church": 6, "Priest": 6, "Social Service": 6,
    "Farmer / Agriculture": 5,
    "Private Sector": 4,
    "Driver": 3, "Courier": 3,
    "Daily wages": 2,
    "Housewife": 1, "Retired": 1, "Armed Forces-Retired": 1, "Student": 1,
}

def age_group(age):
    if pd.isna(age):  return np.nan
    elif age < 30:    return 0
    elif age < 40:    return 1
    elif age < 50:    return 2
    elif age < 60:    return 3
    else:             return 4

# LOAD ACTIVE — duration already in months

df2 = pd.read_excel(
    "../mock-data/Active_Inactive_Non-Diala_Mock.xlsx",
    sheet_name="Active_Users"
)
df2 = df2.dropna(how="all")

for col in df2.select_dtypes(include="object").columns:
    df2[col] = df2[col].str.strip()
    df2[col] = df2[col].replace(replace_vals, np.nan)

df2["Kupp_Occupation"] = df2["Occupation"].map(kupp_map)
df2 = df2.dropna(subset=["Kupp_Occupation"])
df2["Gender"]      = df2["Gender"].map({"M": "Male", "F": "Female"})
df2["Gender_Code"] = df2["Gender"].map({"Male": 1, "Female": 2})
df2["Age_Group"]   = pd.to_numeric(df2["AGE"], errors="coerce").apply(age_group)
df2["HbA1c_BL"]   = pd.to_numeric(df2["HbA1c_BL"], errors="coerce")
df2["HbA1c_FU"]   = pd.to_numeric(df2["HbA1c_FU"], errors="coerce")

# Already in months — just coerce
df2["FU_Duration_months"] = pd.to_numeric(df2["Followup_Duration_BL_FU"], errors="coerce")

df2 = df2[df2["HbA1c_BL"].notna() & df2["HbA1c_FU"].notna()].copy()
df2["Delta_HbA1c"] = df2["HbA1c_FU"] - df2["HbA1c_BL"]

print(f"Active — n with HbA1c pair: {len(df2)}")
print(f"  FU duration available (months): {df2['FU_Duration_months'].notna().sum()}")
print(f"  mean={df2['FU_Duration_months'].mean():.1f}  "
      f"sd={df2['FU_Duration_months'].std():.1f}  "
      f"median={df2['FU_Duration_months'].median():.1f}  "
      f"min={df2['FU_Duration_months'].min():.1f}  "
      f"max={df2['FU_Duration_months'].max():.1f} months")

# LOAD NON-DIALA — subtract dates then convert days → months

df_nd = pd.read_excel(
    "../mock-data/Active_Inactive_Non-Diala_Mock.xlsx",
    sheet_name="Non_DiaLA_Users"
)
df_nd = df_nd.dropna(how="all")

df_nd = df_nd.rename(columns={
    "GENDER":                 "Gender",
    "HDL CHOLESTEROL_BL":     "HDL_BL",
    "SERUM TRIGLYCERIDES_BL": "TGL_BL",
    "HDL CHOLESTEROL_FU":     "HDL_FU",
    "SERUM TRIGLYCERIDES_FU": "TGL_FU",
    "Serum Cholesterol_BL":   "Serum_Cholesterol_BL",
    "Serum Cholesterol_FU":   "Serum_Cholesterol_FU",
})

for col in df_nd.select_dtypes(include="object").columns:
    df_nd[col] = df_nd[col].str.strip()
    df_nd[col] = df_nd[col].replace(replace_vals, np.nan)

df_nd["Kupp_Occupation"] = df_nd["Occupation"].map(kupp_map)
df_nd = df_nd.dropna(subset=["Kupp_Occupation"])
df_nd["Gender"]      = df_nd["Gender"].map({"M": "Male", "F": "Female"})
df_nd["Gender_Code"] = df_nd["Gender"].map({"Male": 1, "Female": 2})
df_nd["Age_Group"]   = pd.to_numeric(df_nd["AGE"], errors="coerce").apply(age_group)
df_nd["HbA1c_BL"]   = pd.to_numeric(df_nd["HbA1c_BL"], errors="coerce")
df_nd["HbA1c_FU"]   = pd.to_numeric(df_nd["HbA1c_FU"], errors="coerce")

# Parse both date columns then subtract → days → months (÷ 30.4375)
df_nd["Baseline_Date"] = pd.to_datetime(df_nd["Baseline_Date"], errors="coerce")
df_nd["FU_DATE"]       = pd.to_datetime(df_nd["FU_DATE"],       errors="coerce")
df_nd["FU_Duration_months"] = (
    (df_nd["FU_DATE"] - df_nd["Baseline_Date"]).dt.days / 30.4375
)

df_nd = df_nd[df_nd["HbA1c_BL"].notna() & df_nd["HbA1c_FU"].notna()].copy()
df_nd["Delta_HbA1c"] = df_nd["HbA1c_FU"] - df_nd["HbA1c_BL"]

print(f"\nNon-DiaLA — n with HbA1c pair: {len(df_nd)}")
print(f"  FU duration available (months): {df_nd['FU_Duration_months'].notna().sum()}")
print(f"  mean={df_nd['FU_Duration_months'].mean():.1f}  "
      f"sd={df_nd['FU_Duration_months'].std():.1f}  "
      f"median={df_nd['FU_Duration_months'].median():.1f}  "
      f"min={df_nd['FU_Duration_months'].min():.1f}  "
      f"max={df_nd['FU_Duration_months'].max():.1f} months")

# Here, built the combined dataframe

shared_cols = [
    "MRNO", "Gender", "Gender_Code", "AGE", "Age_Group",
    "Kupp_Occupation",
    "HbA1c_BL", "HbA1c_FU", "Delta_HbA1c",
    "FU_Duration_months",
]

df_a   = df2[[c for c in shared_cols if c in df2.columns]].copy()
df_nd2 = df_nd[[c for c in shared_cols if c in df_nd.columns]].copy()
df_a["Group"]   = "Active"
df_nd2["Group"] = "Non-DiaLA"

df_dur   = pd.concat([df_a, df_nd2], ignore_index=True)
active   = df_dur[df_dur["Group"] == "Active"]
nondiala = df_dur[df_dur["Group"] == "Non-DiaLA"]

# ANALYSIS 1: Do groups differ in follow-up duration?

print("\n" + "=" * 70)
print("ANALYSIS 1: FOLLOW-UP DURATION — ACTIVE vs NON-DIALA")
print("=" * 70)

a_dur  = active["FU_Duration_months"].dropna()
nd_dur = nondiala["FU_Duration_months"].dropna()

print(f"\n  Active     n={len(a_dur):>6}  mean={a_dur.mean():.1f} months  "
      f"sd={a_dur.std():.1f}  median={a_dur.median():.1f}  "
      f"min={a_dur.min():.1f}  max={a_dur.max():.1f}")
print(f"  Non-DiaLA  n={len(nd_dur):>6}  mean={nd_dur.mean():.1f} months  "
      f"sd={nd_dur.std():.1f}  median={nd_dur.median():.1f}  "
      f"min={nd_dur.min():.1f}  max={nd_dur.max():.1f}")

_, p_a  = stats.shapiro(a_dur.sample(min(len(a_dur),  5000), random_state=42))
_, p_nd = stats.shapiro(nd_dur.sample(min(len(nd_dur), 5000), random_state=42))
u_stat, u_p = stats.mannwhitneyu(a_dur, nd_dur, alternative="two-sided")
u_sig  = "***" if u_p < 0.001 else "**" if u_p < 0.01 else "*" if u_p < 0.05 else "ns"
rbc    = 1 - (2 * u_stat) / (len(a_dur) * len(nd_dur))

print(f"\n  Shapiro Active:    p={p_a:.4f}  ({'normal' if p_a > 0.05 else 'non-normal'})")
print(f"  Shapiro Non-DiaLA: p={p_nd:.4f}  ({'normal' if p_nd > 0.05 else 'non-normal'})")
print(f"\n  Mann-Whitney U={u_stat:.1f}  p={u_p:.4f}  {u_sig}")
print(f"  Rank biserial r={rbc:.3f}")

if u_p < 0.05:
    longer = "Active" if a_dur.mean() > nd_dur.mean() else "Non-DiaLA"
    print(f"\n  ► SIGNIFICANT: {longer} users had significantly longer follow-up")
    print(f"  ► Duration is a potential confounder — control for it in Analysis 3")
else:
    print(f"\n  ► No significant difference in follow-up duration")
    print(f"  ► Duration unlikely to confound outcome comparisons")

# ANALYSIS 2: Does duration predict Delta HbA1c within each group?

print("\n" + "=" * 70)
print("ANALYSIS 2: DOES DURATION PREDICT DELTA HbA1c WITHIN EACH GROUP?")
print("=" * 70)

for grp_name, grp_df in [("Active", active), ("Non-DiaLA", nondiala)]:
    sub = grp_df[["FU_Duration_months", "Delta_HbA1c"]].dropna()
    r, p = stats.spearmanr(sub["FU_Duration_months"], sub["Delta_HbA1c"])
    sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"\n  {grp_name} (n={len(sub)}):  Spearman r={r:.3f}  p={p:.4f}  {sig}")
    if p < 0.05:
        interp = "longer FU → greater improvement" if r < 0 else "longer FU → less improvement"
        print(f"    ► {interp} — duration IS influencing delta in this group")
    else:
        print(f"    ► Duration does not predict HbA1c change in this group")

# ANALYSIS 3: Duration-adjusted ANCOVA

print("\n" + "=" * 70)
print("ANALYSIS 3: DURATION-ADJUSTED DELTA HbA1c (ANCOVA)")
print("  Model: Delta_HbA1c ~ Group + FU_Duration_months")
print("                      + Kupp_Occupation + Age_Group + Gender_Code")
print("=" * 70)

sub = df_dur[["Delta_HbA1c", "Group", "FU_Duration_months",
              "Kupp_Occupation", "Age_Group", "Gender_Code"]].dropna().copy()
sub["Group_bin"] = (sub["Group"] == "Active").astype(int)

model = smf.ols(
    "Delta_HbA1c ~ Group_bin + FU_Duration_months + Kupp_Occupation "
    "+ Age_Group + Gender_Code",
    data=sub
).fit()

ci = model.conf_int()
terms = {
    "Group (Active vs Non-DiaLA)":  "Group_bin",
    "Follow-up duration (months)":  "FU_Duration_months",
    "SES (Kupp score)":             "Kupp_Occupation",
    "Age group":                    "Age_Group",
    "Gender":                       "Gender_Code",
}

print(f"\n  n = {len(sub):,}")
print(f"\n  {'Term':<34} {'Coef':>8}  {'95% CI':>18}  {'p':>8}  {'Sig':>4}")
print(f"  {'-'*34} {'-'*8}  {'-'*18}  {'-'*8}  {'-'*4}")
for label, term in terms.items():
    if term not in model.params: continue
    coef = model.params[term]
    lo   = ci.loc[term, 0]
    hi   = ci.loc[term, 1]
    p    = model.pvalues[term]
    sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"  {label:<34} {coef:>8.3f}  ({lo:.3f}, {hi:.3f})  {p:>8.4f}  {sig}")

print(f"\n  R² = {model.rsquared:.4f}  |  Adj R² = {model.rsquared_adj:.4f}")

group_coef = model.params.get("Group_bin", np.nan)
group_p    = model.pvalues.get("Group_bin", np.nan)
group_sig  = "***" if group_p < 0.001 else "**" if group_p < 0.01 else "*" if group_p < 0.05 else "ns"
print(f"\n  ► After controlling for follow-up duration, SES, age, gender:")
if group_p < 0.05:
    direction = "greater improvement" if group_coef < 0 else "less improvement"
    print(f"    Active users show significantly {direction} in HbA1c vs Non-DiaLA")
    print(f"    (coef={group_coef:.3f}, p={group_p:.4f} {group_sig})")
else:
    print(f"    No significant group difference after adjustment (p={group_p:.4f})")
    print(f"    Follow-up duration may account for some of the raw group difference")


# PLOTS

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ["#5cb85c", "#2E86AB"]

# Plot 1: Duration boxplot
ax = axes[0]
bp = ax.boxplot([a_dur.values, nd_dur.values],
                tick_labels=["Active", "Non-DiaLA"], patch_artist=True)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
y_max = max(np.percentile(a_dur, 95), np.percentile(nd_dur, 95)) * 1.1
ax.set_ylim(top=y_max * 1.15)
ax.plot([1, 2], [y_max, y_max], color="black", linewidth=1)
ax.text(1.5, y_max * 1.02, u_sig, ha="center", fontsize=12, fontweight="bold")
ax.set_title("Follow-up Duration\nActive vs Non-DiaLA", fontweight="bold")
ax.set_ylabel("Follow-up Duration (months)")
ax.set_xlabel("Group")

# Plot 2: Duration vs Delta HbA1c — Active
ax = axes[1]
sub_a = active[["FU_Duration_months", "Delta_HbA1c"]].dropna()
ax.scatter(sub_a["FU_Duration_months"], sub_a["Delta_HbA1c"],
           alpha=0.15, s=8, color=colors[0])
m, b = np.polyfit(sub_a["FU_Duration_months"], sub_a["Delta_HbA1c"], 1)
x_line = np.linspace(sub_a["FU_Duration_months"].min(),
                     sub_a["FU_Duration_months"].max(), 100)
ax.plot(x_line, m * x_line + b, color=colors[0], linewidth=2)
r, p = stats.spearmanr(sub_a["FU_Duration_months"], sub_a["Delta_HbA1c"])
sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8, alpha=0.6)
ax.set_title(f"Duration vs Δ HbA1c — Active\nr={r:.3f} {sig}", fontweight="bold")
ax.set_xlabel("Follow-up Duration (months)")
ax.set_ylabel("Δ HbA1c (FU - BL)")

# Plot 3: Duration vs Delta HbA1c — Non-DiaLA
ax = axes[2]
sub_nd = nondiala[["FU_Duration_months", "Delta_HbA1c"]].dropna()
ax.scatter(sub_nd["FU_Duration_months"], sub_nd["Delta_HbA1c"],
           alpha=0.15, s=8, color=colors[1])
m, b = np.polyfit(sub_nd["FU_Duration_months"], sub_nd["Delta_HbA1c"], 1)
x_line = np.linspace(sub_nd["FU_Duration_months"].min(),
                     sub_nd["FU_Duration_months"].max(), 100)
ax.plot(x_line, m * x_line + b, color=colors[1], linewidth=2)
r, p = stats.spearmanr(sub_nd["FU_Duration_months"], sub_nd["Delta_HbA1c"])
sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8, alpha=0.6)
ax.set_title(f"Duration vs Δ HbA1c — Non-DiaLA\nr={r:.3f} {sig}", fontweight="bold")
ax.set_xlabel("Follow-up Duration (months)")
ax.set_ylabel("Δ HbA1c (FU - BL)")

plt.suptitle("Follow-up Duration Analysis — Active vs Non-DiaLA",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("followup_duration_analysis.png", dpi=150, bbox_inches="tight")
plt.close()